In [2]:
import warnings
from langchain._api import LangChainDeprecationWarning

warnings.simplefilter("ignore", category=LangChainDeprecationWarning)

In [1]:
import os
from dotenv import load_dotenv,find_dotenv
_ = load_dotenv(find_dotenv())

groq_api_key =  os.environ["GROQ_API_KEY"]

In [3]:
from langchain_groq import ChatGroq
llmModel = ChatGroq(temperature=0,model = "llama3-70b-8192")

In [4]:
#import bs4
#from langchain import hub
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import HumanMessagePromptTemplate
from langchain_core.prompts import PromptTemplate

In [12]:
loader = TextLoader("data/be-good.txt")

docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)

splits = splitter.split_documents(docs)

vector_db = Chroma.from_documents(documents=splits, embedding=HuggingFaceEmbeddings())

retriever =  vector_db.as_retriever()

prompt = ChatPromptTemplate.from_messages([
    HumanMessagePromptTemplate(
        prompt=PromptTemplate(
            input_variables=["context", "question"],
            template=(
                "You are an assistant for question-answering tasks. "
                "Use the following pieces of retrieved context to answer the question. "
                "If you don't know the answer, just say that you don't know. "
                "Use three sentences maximum and keep the answer concise.\n"
                "Question: {question} \nContext: {context} \nAnswer:"
            )
        )
    )
])

def format_docs(doc):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = ({"context": retriever | format_docs, "question":RunnablePassthrough()}
            |prompt
            |llmModel
            |StrOutputParser()
)

rag_chain.invoke("What is this article about?")

'This article discusses the idea that successful startups often behave like charities, prioritizing the needs of their users and focusing on making something people want, rather than solely on making money. The author argues that being benevolent can lead to success, as it improves morale, makes others want to help, and provides a compass for decision-making.'

In [13]:
rag_chain.invoke("What is a succesful startup?")

'A successful startup is one that "makes something people want" and "don\'t worry too much about making money." This approach can lead to a successful startup that is benevolent and helps people, which in turn can attract users, investors, and employees.'